# SQL Practice - Day 3

## Q1

**Task:** For each customer, find their latest order.

In [1]:
import pandas as pd
from pandasql import sqldf
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-01-10','2025-01-20','2025-02-01'],'amount':[500,700,300,1000,200,800]})
df

,order_id,customer_id,order_date,amount
0,1,101,2025-01-01,500
1,2,101,2025-01-15,700
2,3,102,2025-01-05,300
3,4,103,2025-01-10,1000
4,5,102,2025-01-20,200
5,6,101,2025-02-01,800


In [3]:
sqldf("""
        select customer_id , amount , max(order_date) as lates_order
        from df
        group by customer_id



""")

,customer_id,amount,lates_order
0,101,800,2025-02-01
1,102,200,2025-01-20
2,103,1000,2025-01-10


## Q2

**Task:** Find the top 2 customers by total spending.

In [4]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6,7],'customer_id':[101,101,102,103,102,101,104],'amount':[500,700,300,1000,200,800,1500]})
df

,order_id,customer_id,amount
0,1,101,500
1,2,101,700
2,3,102,300
3,4,103,1000
4,5,102,200
5,6,101,800
6,7,104,1500


In [8]:
sqldf("""
        select customer_id , sum(amount) as toatl_spending
        from df
        group by customer_id
        order by toatl_spending desc
        limit 2 



""")

,customer_id,toatl_spending
0,101,2000
1,104,1500


## Q3

**Task:** Using a CTE, find departments where highest salary > company average salary.

In [9]:
import pandas as pd
df=pd.DataFrame({'department':['HR','HR','HR','IT','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,40000,45000,80000,90000,85000]})


,department,salary
0,HR,50000
1,HR,60000
2,HR,70000
3,IT,40000
4,IT,45000
5,IT,80000
6,Sales,90000
7,Sales,85000


np.float64(65000.0)

In [16]:
sqldf("""
        with my_cte as (
        select department, max(salary) as highest_salary
        from df
        group by department
        )
        select * from my_cte
        where highest_salary > (select avg(salary) 
                                from df)

""")

,department,highest_salary
0,HR,70000
1,IT,80000
2,Sales,90000


## Q4

**Task:** Find the 3rd highest salary in each department.

In [17]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6,7,8],'name':['A','B','C','D','E','F','G','H'],'department':['HR','HR','HR','HR','IT','IT','IT','IT'],'salary':[50000,60000,70000,55000,40000,45000,80000,70000]})
df

,emp_id,name,department,salary
0,1,A,HR,50000
1,2,B,HR,60000
2,3,C,HR,70000
3,4,D,HR,55000
4,5,E,IT,40000
5,6,F,IT,45000
6,7,G,IT,80000
7,8,H,IT,70000


In [27]:
sqldf("""
        select name , department,salary
        from (select name , department , salary,
                dense_rank()
                over(partition by department order by salary desc) as rn
               from df)t
        where rn = 3


""")

,name,department,salary
0,D,HR,55000
1,F,IT,45000


## Q5

**Task:** Find products with price above category average but below category maximum.

In [28]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,150,200,250,300,350]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350


In [31]:
sqldf("""SELECT product_id
FROM (
    SELECT product_id,
           price,
           AVG(price) OVER (PARTITION BY category) AS avg_price,
           MAX(price) OVER (PARTITION BY category) AS max_price
    FROM df
) t
WHERE price > avg_price
  AND price < max_price;
""")

,product_id


## Q6

**Task:** Return customer name and total spending including customers with no orders.

In [32]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'customer_name':['Alice','Bob','Charlie','David']})
df1=pd.DataFrame({'order_id':[1,2,3,4],'customer_id':[101,101,102,103],'amount':[500,700,300,1000]})
df,df1

(   customer_id customer_name
 0          101         Alice
 1          102           Bob
 2          103       Charlie
 3          104         David,
    order_id  customer_id  amount
 0         1          101     500
 1         2          101     700
 2         3          102     300
 3         4          103    1000)

In [44]:
sqldf("""
        select c.customer_name as customer_name , o.amount as total_spending
        from df c
        left join df1 o on c.customer_id   = o.customer_id  
        group by c.customer_id



""")

,customer_name,total_spending
0,Alice,500.0
1,Bob,300.0
2,Charlie,1000.0
3,David,NaN


## Q7

**Task:** Find employees who earn more than their manager.

In [45]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'manager_id':[None,1,1,2,2],'salary':[100000,60000,120000,70000,50000]})
df

,emp_id,name,manager_id,salary
0,1,A,NaN,100000
1,2,B,1.0,60000
2,3,C,1.0,120000
3,4,D,2.0,70000
4,5,E,2.0,50000


In [48]:
sqldf("""
        select e.name as emp_name, e.salary as emp_salary , m.name as manager_name , m.salary as manager_salary
        from df e
        join df m on e.manager_id = m.emp_id 
        where e.salary > m.salary


""")

,emp_name,emp_salary,manager_name,manager_salary
0,C,120000,A,100000
1,D,70000,B,60000


## Q8

**Task:** For each product show previous sale amount.

In [55]:
import pandas as pd
df=pd.DataFrame({'sale_id':[1,2,3,4,5],'product_id':['A','A','A','B','B'],'sale_date':['2025-01-01','2025-01-02','2025-01-03','2025-01-01','2025-01-02'],'amount':[100,150,120,200,250]})
df

,sale_id,product_id,sale_date,amount
0,1,A,2025-01-01,100
1,2,A,2025-01-02,150
2,3,A,2025-01-03,120
3,4,B,2025-01-01,200
4,5,B,2025-01-02,250


In [56]:
sqldf("""
        SELECT
    product_id,
    sale_date,
    amount,
    LAG(amount) OVER (
        PARTITION BY product_id
        ORDER BY sale_date
    ) AS previous_sale_amount
FROM df;


""")

,product_id,sale_date,amount,previous_sale_amount
0,A,2025-01-01,100,NaN
1,A,2025-01-02,150,100.0
2,A,2025-01-03,120,150.0
3,B,2025-01-01,200,NaN
4,B,2025-01-02,250,200.0


## Q9

**Task:** Using a CTE, find customers above average customer spending.

In [65]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'amount':[500,700,300,1000,200,800]})
df

,order_id,customer_id,amount
0,1,101,500
1,2,101,700
2,3,102,300
3,4,103,1000
4,5,102,200
5,6,101,800


In [69]:
sqldf("""
        with my_cte as (
        select customer_id ,sum(amount) as avg_spending
        from df
        group by customer_id
        having sum(amount) > (select avg(amount) 
                                from (select sum(amount) as amount
                                        from df
                                        group by customer_id))
        )
        select * from my_cte


""")

,customer_id,avg_spending
0,101,2000


## Q10

**Task:** Find the 2nd highest priced product in each category.

In [70]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6,7,8],'category':['A','A','A','B','B','B','C','C'],'price':[100,150,200,250,300,350,500,450]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350
6,7,C,500
7,8,C,450


In [71]:
sqldf("""
    with my_cte as (
    select category , price
    from (select category , price,
            dense_rank()
            over(partition by category order by price desc) as rn
          from df) t
    where rn = 2
    
    )
    select * from my_cte



""")

,category,price
0,A,150
1,B,300
2,C,450
